## Final Data Preparation

This Jupyter Notebook deletes rare or not plausible combinations of Land Cover and Lanc Use. It was agreed on this after a team-meeting and exchange with Dr. Hobley. Goal is to have a dataset as clean as possible to use in Machine Learning. Threshold: <30, see Folder: 02_notes = image that illustrates this step.

OUTPUT: df_SHI_with_LC_LU_more_reduced.csv

This output is then passed on to the other group members. It serves as the basis for adding additional independent variables: temperature, elevation, Köppen-Geiger climate classes, and precipitation. It cannot be ruled out that some more data points may be omitted during these following steps, resulting in a reduced final dataset.

_Note: exactly this happened and the dataset used in ML had a size of 4467 (gbm) and 4426 (cforest) points. For cforest the Köppen-Geiger-Climate-Classes had to be reduced even further, see cforest_shi.md for further details._

In [2]:
import pandas as pd
import csv

In [3]:
# load data
df = pd.read_csv("03_output/df_SHI_with_LC_LU_reduced.csv")

# check data
df.head()
print(len(df))

print("Landcover:", df["Landcover"].unique())
print("Landuse:", df["Landuse"].unique())

4538
Landcover: <StringArray>
['Cropland', 'Woodland', 'Grassland', 'Shrubland', 'Bareland']
Length: 5, dtype: str
Landuse: <StringArray>
['Agriculture (excluding fallow land and kitchen gardens)',
                                                'Forestry',
                                             'Fallow land',
               'Semi-natural and natural areas not in use']
Length: 4, dtype: str


In [4]:
# set a threshold limit for unplausible or rare combinations of LC and LU
threshold = 30

# count and print these rare combinations
combo_counts = (
    df.groupby(["Landcover", "Landuse"])
      .size()
      .reset_index(name="count")
)

rare = combo_counts[combo_counts["count"] < threshold]
total_rare = rare["count"].sum()


print(f"Rare combinations of Landcover/Landuse with <= {threshold} counts")
print("-" * 71)
print(rare)
print("-" * 71)
print(f"Sum: {total_rare} to be dropped")
print("-" * 71)


# compute frequency of each Landcover/Landuse combination and keep only rows above or equal to threshold 30
mask = df.groupby(["Landcover", "Landuse"])["Landcover"].transform("size") >= threshold
df_clean = df[mask].copy()

# check the differences in dataset size
print(f"{'Dataset size before:':<35} {len(df):>6}")
print(f"{'Dataset size after cleaning:':<35} {len(df_clean):>6}")
print(f"{'Rows dropped:':<35} {total_rare:>6}")
print("-" * 71)

Rare combinations of Landcover/Landuse with <= 30 counts
-----------------------------------------------------------------------
    Landcover                                            Landuse  count
2    Bareland                                           Forestry      2
3    Bareland          Semi-natural and natural areas not in use      2
5    Cropland                                           Forestry      1
8   Grassland                                           Forestry     21
11  Shrubland                                        Fallow land      2
12  Shrubland                                           Forestry     10
14   Woodland  Agriculture (excluding fallow land and kitchen...     18
-----------------------------------------------------------------------
Sum: 56 to be dropped
-----------------------------------------------------------------------
Dataset size before:                  4538
Dataset size after cleaning:          4482
Rows dropped:                           56


In [5]:
# save the df in output folder
df_clean.to_csv("03_output/df_SHI_with_LC_LU_more_reduced.csv", index=False)